In [2]:
from graph_tool.all import *
import numpy as np

# Susceptible: 0
# Infectado: 1
# Debilitado: 2
# Recuperado: 3

In [45]:
# número de nodos
n = 100
# grado medio
z = 8
# probabilidad de que entre dos nodos escogidos aleatoriamente exista una arista
p = z/n
# densidad inicial de nodos infectados
rho0 = 0.5
# probabilidad de que un nodo susceptible se infecte
kappa= 0.2
# probabilidad de que un nodo susceptible se debilite
mu = 0.1
#probabilidad de que un nodo debilitado se infecte
eta = 0.1
# probabilidad de que un nodo susceptible no cambie de estado al interactuar con un nodo infectado
ss = 1 - mu - kappa
# probabilidad de que un nodo debilitado no cambie de estado al interactuar con un nodo infectado
ww = 1 - eta

# inicialización del grafo erdos renyi
g = random_graph(n, lambda: np.random.poisson((n-1) * p), directed=False, model="erdos")

# declarar una propiedad llamada "estado" que va a guardar el estado de cada nodo
estado = g.new_vertex_property("short")

# Asignar a todos los nodos el estado susceptible
estado.get_array()[:] = 0

# Generar rho0*n índices aleatorios
infected_index = np.random.choice(np.arange(0, n), size=int(n*rho0), replace=False)

# Actualizar el estado de rho0*n nodos al estado infectado
estado.get_array()[infected_index] = 1

# Iterador que contiene los nodos infectados
I = (g.vertex(i) for i in infected_index)

In [47]:
# Mejorar con las ideas de la documentación de graph_tool:
# Fast iteration over vertices and edges

def reaction(vertex):
    """
    Función que realiza un paso de tiempo para un nodo infectado:
    vertex es un nodo en estado infectado.
    """
    # inicia una lista que guarda los nodos que serán infectados
    new_infected = []
    # actualiza el estado del nodo actual infectado a recuperado
    estado[vertex] = 3
    # obtiene los vecinos del nodo infectado
    vecinos = vertex.out_neighbors()
    # obtiene un iterador de los nodos en estado susceptible
    S = (v for v in vecinos if estado[v]==0)
    # obtiene un iterador de los nodos en estado debilitado
    W = (v for v in vecinos if estado[v]==2)
    # itera sobre los susceptibles
    for s in S:
        # calcula el nuevo estado del nodo susceptible
        new_state = np.random.choice([0, 1, 2], size=1, p=[ss, kappa, mu])
        # asigna el nuevo estado del nodo susceptible
        estado[s] = new_state
        # revisa si el nuevo estado es infectado para añadirlo a la lista que guarda la lista de los infectados
        if new_state == 1:
            new_infected.append(s)
    # itera sobre los debilitados
    for w in W:
        #calcula el nuevo estado del nodo debilitado
        new_state = np.random.choice([2, 1], size=1, p=[ww, eta])
        #asigna el nuevo estado del nodo debilitado
        estado[w] = new_state
        # revisa si el nuevo estado es infectado para añadirlo a la lista que guarda la lista de los infectados
        if new_state == 1:
            new_infected.append(s)
    # devuelve los vecinos que se infectaron
    return new_infected

In [53]:
# revisa si hay nodos infectados
while I:
    # inicia una lista que guarda los nodos que serán infectados en el paso de tiempo n
    new_infected_n = []
    # itera sobre los nodos infectados
    for i in I:
        # hace que el nodo infectado reaccione con sus vecinos
        ni = reaction(i)
        # agrega los vecinos infectados a la lista de los nuevos infectados
        new_infected_n.append(ni)
    # actualiza la lista de los nuevos infectados para el paso de tiempo n+1
    I = new_infected_n